In [48]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field
from typing import Literal

## Libs for Agents
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_community.utilities import OpenWeatherMapAPIWrapper

In [49]:
class FlowState(BaseModel):
    question:str = Field(description="User Asked Question")
    category : Literal['coding', 'google_search', 'weather'] = Field(default="google_search")
    answer : str = Field(default="")

In [50]:
class QuestionCategory(BaseModel):
    category : Literal['coding', 'google_search', 'weather'] = Field(default="google_search",description="Question Category")
    


In [51]:
llm=ChatGroq(model="openai/gpt-oss-20b")


In [52]:
def check_category(State:FlowState):
    st_llm=llm.with_structured_output(QuestionCategory)
    res=st_llm.invoke(f"I want to know the category of my question ,question is {State.question},If not sure give 'google_search'")
    State.category=res.category
    return State

In [ ]:
#google seacrh agent
search=GoogleSerperAPIWrapper()
search_tool=[search.run]

search_agent=create_agent(
    model=llm,
    tools=search_tool,
    system_prompt="You are an agent that search anything from google"
)


#weather api tool
# weather=OpenWeatherMapAPIWrapper()
# weather_tools=[weather.run]

#temp_test_tool
@tool
def get_weather_tool():
    
    '''
    a weather tool
    '''
    res="The cureent temp is 33C rn"
    return res

#weather_agent

weather_agent=create_agent(
    model=llm,
    tools=[get_weather_tool],
    system_prompt="You are an weather fetching agent who fetchs current weather details at required location"
)

#coding agent

@tool
def get_coding():
    """"
        you are an expert coder:
    """"



In [56]:
from email import message
result = weather_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the temperature in Kolkata right now?"
        }
    ]
})

print(result["messages"][-1].content)

The current temperature in Kolkata is **33 °C**.
